# Day 4 — Solution: Skewness & Kurtosis

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices(["SPY", "TLT", "XLE"], start="2005-01-01")
else:
    px = synthetic_prices(n_days=5000, n_assets=3, seed=34, drift_spread=0.0003)
    px.columns = ["SPY", "TLT", "XLE"]

## E1 — moments with error bars

In [ ]:
for c in px.columns:
    s = px[c].pct_change().dropna()
    z = (s - s.mean()) / s.std()
    n = len(s)
    print(f"{c}: skew {z.skew():+.2f} ± {np.sqrt(6/n):.2f} | "
          f"kurt {(z**4).mean()-3:.1f} ± {np.sqrt(24/n):.1f} (n={n})")

**Expected (real data):** all three clear 2 SE on kurtosis by wide
margins (SPY ~10–15, XLE ~8–15, TLT ~3–8 — window-dependent); skew is
negative and 2+ SE for SPY/XLE, weaker for TLT. **Every asset in your
table is statistically non-normal by an order of magnitude — the
interesting question is only "how much", which is why we report the
effect sizes, not the p-values.**

## E2 — the k⁴/n law

In [ ]:
rng = np.random.default_rng(4)
base = rng.standard_normal(5000)
for k in [5, 10, 20]:
    x = np.append(base, k)
    z = (x - x.mean()) / x.std()
    print(f"k={k:2d}: excess kurt {((z**4).mean()-3):8.1f} vs k⁴/n rule {k**4/5001:.1f}")
zs = px["SPY"].pct_change().dropna()
zr = (zs - zs.mean()) / zs.std()
print(f"SPY worst z: {zr.min():.1f}; k⁴/n at that z: {zr.min()**4/len(zr):.1f}")

**Expected reasoning.** The k⁴/n rule tracks the simulated values
closely (5→0.06, 10→2, 20→32 vs the rule 0.06/2.0/32.0). SPY's worst z
since 2005 is ~−6 to −8: one day contributes 6⁴/5000 ≈ 0.3 to 8⁴/5000
≈ 0.8 — NOT dominant at n=5,000. **But at n = 252 (one year), that
same day would contribute 8⁴/252 ≈ 16 — the entire kurtosis estimate.
Short-window kurtosis IS its worst day.** That's why year-by-year
kurtosis of the same asset swings wildly.

## E3 — the three QQ patterns

In [ ]:
rng = np.random.default_rng(5)
n = 5000
samples = {
    "normal": rng.standard_normal(n),
    "t(5)": rng.standard_t(5, n) / np.sqrt(5/3),
    "skewed fat-tail": np.concatenate([rng.normal(0, 1, int(0.7*n)),
                                        -np.abs(rng.standard_t(3, int(0.3*n)))*2]),
}
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (nm, x) in zip(axes, samples.items()):
    z = np.sort((x - x.mean()) / x.std())
    q = st.norm.ppf((np.arange(n) + 0.5) / n)
    ax.scatter(q, z, s=3); ax.plot([-4, 4], [-4, 4], "r--")
    ax.set_title(nm); ax.set_xlim(-4, 4); ax.set_ylim(-6, 6)
plt.tight_layout(); plt.show()

**Expected reading.** (a) hugs the line. (b) both ends curl away
symmetrically — κ>0, ν≈0. (c) left end plunges below, right end
approximately tracks — ν<0 with κ>0: the equity-market signature.
**Location is diagnosis: which end, and where the bend starts, tells
you which moment is misbehaving and at what probability it begins.**

## E4 — real shapes, matched

In [ ]:
for c in px.columns:
    s = px[c].pct_change().dropna()
    z = np.sort((s - s.mean()) / s.std())
    q = st.norm.ppf((np.arange(len(z)) + 0.5) / len(z))
    plt.plot(q, z, label=c)
plt.plot([-4, 4], [-4, 4], "k--", lw=0.8); plt.legend(); plt.show()

**Expected matches:** SPY ≈ skewed-t (left tail breaks ~z=−2.5);
XLE ≈ t(4–5) with negative skew (worse both tails); TLT ≈ t(6–8)
(bonds' tails are real but thinner; 2020 fattens any window containing
it). **The defense sentence names the bend location and the moment:
"XLE matches a t(4) in the tails with skew −0.5 — normal VaR
understates its 1% tail by ~3×."**

## E5 — strategy autopsy (exemplar)

Short-vol: 14 months of +1.2%, 4 months of −0.5%, 1 month of −12%, 1
month of +0.3% → mean ≈ +0.05%/mo (barely positive), median +1.2%,
skew ≈ −1.2, kurt ≈ 6. Long-vol trend: 9 months of −1%, 5 months of
+4% → mean ≈ +0.55%/mo, median −1%, skew ≈ +0.7, kurt ≈ 2. At n = 12:
SE(skew) = 0.71, SE(kurt) = 1.41 — **neither moment is measurable in a
year**; the win rates (92% vs 42%) separate faster (SE of a 0.9 rate
gap at n=12 ≈ 0.14 → 6+ SE). "We'll know in a year" is true only for
win rate — which is exactly the statistic that flatters the short-vol
book. The moments you NEED are the slowest to arrive.